In [0]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Dataset root — change here if the path ever moves
DATA_ROOT = "/Volumes/tomato_data/default/raw/tomato/"

# Image extensions we consider valid
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Dataset root: {DATA_ROOT}")
print(f"Exists: {os.path.exists(DATA_ROOT)}")

In [0]:
def print_tree(root, max_depth=3):
    root = Path(root)
    if not root.exists():
        print(f"PATH DOES NOT EXIST: {root}")
        return
    base_depth = len(root.parts)
    for path in sorted(root.rglob("*")):
        depth = len(path.parts) - base_depth
        if depth > max_depth:
            continue
        indent = "    " * depth
        marker = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{marker}")

print_tree(DATA_ROOT, max_depth=2)

In [0]:
records = []

for split_dir in sorted(Path(DATA_ROOT).iterdir()):
    if not split_dir.is_dir():
        continue
    split_name = split_dir.name
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        count = sum(
            1 for f in class_dir.iterdir()
            if f.is_file() and f.suffix.lower() in IMG_EXTS
        )
        records.append({
            "split": split_name,
            "class": class_name,
            "count": count,
        })

df_counts = pd.DataFrame(records)

if df_counts.empty:
    print("No images found. Check DATA_ROOT and folder structure.")
else:
    pivot = df_counts.pivot(index="class", columns="split", values="count").fillna(0).astype(int)
    print("Image counts per class per split:")
    print(pivot)
    print()
    print("Totals per split:")
    print(df_counts.groupby("split")["count"].sum())
    print()
    print(f"Grand total: {df_counts['count'].sum()} images")
    print(f"Classes found: {sorted(df_counts['class'].unique())}")

In [0]:
sample_records = []
SAMPLES_PER_CLASS = 5

for split_dir in sorted(Path(DATA_ROOT).iterdir()):
    if not split_dir.is_dir():
        continue
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        images = [f for f in class_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in IMG_EXTS]
        for img_path in images[:SAMPLES_PER_CLASS]:
            try:
                with Image.open(img_path) as im:
                    sample_records.append({
                        "split": split_dir.name,
                        "class": class_dir.name,
                        "width": im.width,
                        "height": im.height,
                        "mode": im.mode,
                        "format": im.format,
                    })
            except Exception as e:
                sample_records.append({
                    "split": split_dir.name,
                    "class": class_dir.name,
                    "width": None,
                    "height": None,
                    "mode": f"ERROR: {e}",
                    "format": None,
                })

df_img = pd.DataFrame(sample_records)

print("Sampled image properties:")
print(df_img.head(15).to_string(index=False))
print()
print("Size distribution (width x height):")
print(df_img.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))
print()
print("Color mode distribution:")
print(df_img["mode"].value_counts())
print()
print("Format distribution:")
print(df_img["format"].value_counts())

In [0]:
classes = sorted(df_counts["class"].unique())
n_classes = len(classes)
n_cols = 6
n_rows = min(3, n_classes)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
if n_rows == 1:
    axes = np.array([axes])

train_dir = Path(DATA_ROOT) / "train"
if not train_dir.exists():
    # Fall back to first split found
    train_dir = next(d for d in Path(DATA_ROOT).iterdir() if d.is_dir())

for row in range(n_rows):
    for col in range(n_cols):
        ax = axes[row][col]
        ax.axis("off")
        if col >= n_classes:
            continue
        class_name = classes[col]
        class_dir = train_dir / class_name
        if not class_dir.exists():
            continue
        images = [f for f in class_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in IMG_EXTS]
        if not images:
            continue
        img_path = images[row % len(images)]
        with Image.open(img_path) as im:
            ax.imshow(im.convert("RGB"))
            ax.set_title(class_name, fontsize=9)

plt.suptitle("Sample images per class (rows = different samples)", fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
if not df_counts.empty:
    pivot = df_counts.pivot(index="class", columns="split", values="count").fillna(0)
    pivot.plot(kind="bar", figsize=(12, 5))
    plt.title("Image count per class per split")
    plt.ylabel("Number of images")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [0]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

if not df_counts.empty:
    total = df_counts["count"].sum()
    n_classes = df_counts["class"].nunique()
    splits = sorted(df_counts["split"].unique())

    print(f"Total images:      {total}")
    print(f"Number of classes: {n_classes}")
    print(f"Splits:            {', '.join(splits)}")
    print()
    print("Per-class totals (all splits):")
    print(df_counts.groupby("class")["count"].sum().sort_values(ascending=False))
    print()

    if not df_img.empty and df_img["width"].notna().any():
        widths = df_img["width"].dropna().unique()
        heights = df_img["height"].dropna().unique()
        modes = df_img["mode"].dropna().unique()
        print(f"Unique image sizes: {len(widths)} widths x {len(heights)} heights")
        print(f"Color modes seen:   {list(modes)}")

        if len(widths) == 1 and len(heights) == 1:
            print("→ All images same size. Resize step can be skipped (still normalize).")
        else:
            print("→ Mixed image sizes. Resize to 224x224 in transforms.")

        if set(modes) != {"RGB"}:
            print("→ Non-RGB images present. Convert to RGB in transforms.")
        else:
            print("→ All images RGB. No conversion needed.")
else:
    print("No data to summarize. Fix DATA_ROOT and re-run.")

